# Distilling a 0.8B Tool-Calling Agent

In this notebook, we are going to take a tiny model and ask a very concrete question:

> Can `Qwen/Qwen3.5-0.8B` become more reliable at multi-turn tool calling after sequence-level distillation from a stronger teacher?

This is not a general chat-model benchmark. We are treating the small model as an action policy inside a tool loop. It sees a user request and available tools, emits a tool call, receives the tool result, and keeps going until the task is done.

## The Shape Of The Experiment

We will move through one full loop:

1. Load a small slice of BFCL, a benchmark for function and tool calling.
2. Run the base 0.8B student model and inspect where it fails.
3. Ask a stronger teacher model to produce successful tool-use trajectories.
4. Fine-tune the student on those teacher trajectories using normal next-token training.
5. Run the same benchmark slice again and compare the behavior.

The important concept is that a benchmark is not just a dataset. For tool use, the benchmark is a contract: data, prompt format, parser, environment, and scorer all have to agree.

## What We Are Not Doing Yet

This first notebook keeps the harness visible. We will use plain Python before bringing in orchestration frameworks like LangGraph or structured-output helpers like BAML.

That is intentional. If the small model emits an invalid tool call, we want to see the invalid tool call. Later notebooks can ask how engineering scaffolding changes reliability. This notebook asks what the model learns from distillation.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import json

ROOT = Path.cwd()
if not (ROOT / "docs" / "plan.md").exists() and (ROOT.parent / "docs" / "plan.md").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"

STUDENT_MODEL = "Qwen/Qwen3.5-0.8B"
TEACHER_MODEL = "mlx-community/Qwen3.5-35B-A3B-4bit"

BFCL_FILE = "BFCL_v3_multi_turn_base.json"

print(f"Project root: {ROOT}")
print(f"Student: {STUDENT_MODEL}")
print(f"Teacher: {TEACHER_MODEL}")

Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Student: Qwen/Qwen3.5-0.8B
Teacher: mlx-community/Qwen3.5-35B-A3B-4bit


## First Mental Model: The Harness

A normal text benchmark can look like this:

`question -> model -> final answer -> string comparison`

A tool-use benchmark looks more like this:

`state -> prompt -> model action -> parse action -> execute tool -> new state -> repeat -> score`

That loop is the harness. The model is only one part of the system. The environment and scorer are what make the task measurable.

## Two Views Of The Same Loop

There are two useful ways to look at the system.

The engineering view separates the pieces so we can debug them. The model proposes actions. The harness renders prompts, parses tool calls, controls the loop, and logs what happened. The simulator owns the hidden world state and returns observations when tools are executed. The scorer turns the outcome into metrics or reward.

![Engineering view of model, harness, simulator, and scorer boundaries](assets/harness-boundaries.png)

The RL view wraps most of those pieces into one larger object: the environment. From that angle, the policy is the model, and everything outside the model is the environment. The model emits an action, the environment returns an observation and reward, and the optimizer uses that signal to update the model.

Both views are correct. The engineering view is better for building and debugging. The RL view is better for thinking about training.

![RL view of policy, environment, reward, and optimizer loop](assets/rl-environment-loop.png)

## A Tiny Tool-Use Benchmark

Before we touch BFCL, we will build the smallest possible version of the same idea.

The task is: create a file with a specific path and content. The model's action is a structured tool call. The simulator is a fake file system. The scorer checks the final file-system state.

This toy benchmark is not the research result. It is here to make the moving parts visible.

In [2]:
@dataclass
class ToyTask:
    user_request: str
    expected_path: str
    expected_content: str


@dataclass
class ToolResult:
    ok: bool
    observation: str


class ToyFileSystem:
    def __init__(self) -> None:
        self.files: dict[str, str] = {}

    def write_file(self, path: str, content: str) -> ToolResult:
        self.files[path] = content
        return ToolResult(ok=True, observation=f"Wrote {path}")

    def snapshot(self) -> dict[str, str]:
        return dict(self.files)


toy_task = ToyTask(
    user_request="Create a note called plan.txt with the content hello from the harness.",
    expected_path="plan.txt",
    expected_content="hello from the harness",
)

toy_env = ToyFileSystem()
toy_env.snapshot()

{}

The important boundary is that the model does not directly mutate the file system. It only proposes an action. The harness parses that action, decides whether it is valid, executes it, and then scores what happened.

In [3]:
def render_toy_prompt(task: ToyTask, env: ToyFileSystem) -> str:
    tools = [
        {
            "name": "write_file",
            "description": "Write text content to a file path.",
            "parameters": {
                "path": "string",
                "content": "string",
            },
        }
    ]
    return json.dumps(
        {
            "user_request": task.user_request,
            "available_tools": tools,
            "current_files": env.snapshot(),
            "instruction": "Return exactly one tool call as JSON with name and arguments.",
        },
        indent=2,
    )


def parse_json_tool_call(raw_text: str) -> tuple[dict[str, Any] | None, list[str]]:
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        return None, [f"Invalid JSON: {exc}"]

    errors: list[str] = []
    if not isinstance(parsed, dict):
        errors.append("Tool call must be a JSON object.")
    if parsed.get("name") != "write_file":
        errors.append("Tool call name must be write_file.")

    arguments = parsed.get("arguments")
    if not isinstance(arguments, dict):
        errors.append("Tool call arguments must be an object.")
    else:
        if not isinstance(arguments.get("path"), str):
            errors.append("arguments.path must be a string.")
        if not isinstance(arguments.get("content"), str):
            errors.append("arguments.content must be a string.")

    return parsed if not errors else None, errors


def execute_json_tool_call(env: ToyFileSystem, tool_call: dict[str, Any]) -> ToolResult:
    arguments = tool_call["arguments"]
    return env.write_file(path=arguments["path"], content=arguments["content"])


def score_toy_task(task: ToyTask, env: ToyFileSystem) -> dict[str, Any]:
    actual_content = env.files.get(task.expected_path)
    return {
        "success": actual_content == task.expected_content,
        "expected_path": task.expected_path,
        "expected_content": task.expected_content,
        "actual_content": actual_content,
        "final_files": env.snapshot(),
    }

Now we can run the full harness with a fake model. The fake model is just a Python function for now. Later, this exact slot becomes `Qwen/Qwen3.5-0.8B`.

In [4]:
def good_fake_model(prompt: str) -> str:
    return json.dumps(
        {
            "name": "write_file",
            "arguments": {
                "path": "plan.txt",
                "content": "hello from the harness",
            },
        }
    )


def bad_fake_model(prompt: str) -> str:
    return "Done, I created the note."


def run_toy_harness(task: ToyTask, model_fn) -> dict[str, Any]:
    env = ToyFileSystem()
    prompt = render_toy_prompt(task, env)
    raw_model_output = model_fn(prompt)
    tool_call, parse_errors = parse_json_tool_call(raw_model_output)

    tool_result = None
    if tool_call is not None:
        tool_result = execute_json_tool_call(env, tool_call)

    return {
        "prompt": prompt,
        "raw_model_output": raw_model_output,
        "parse_errors": parse_errors,
        "tool_result": tool_result,
        "score": score_toy_task(task, env),
    }


good_run = run_toy_harness(toy_task, good_fake_model)
bad_run = run_toy_harness(toy_task, bad_fake_model)

print("Good fake model score:")
print(json.dumps(good_run["score"], indent=2))
print("\nBad fake model parse errors:")
print(json.dumps(bad_run["parse_errors"], indent=2))
print("\nBad fake model score:")
print(json.dumps(bad_run["score"], indent=2))

Good fake model score:
{
  "success": true,
  "expected_path": "plan.txt",
  "expected_content": "hello from the harness",
  "actual_content": "hello from the harness",
  "final_files": {
    "plan.txt": "hello from the harness"
  }
}

Bad fake model parse errors:
[
  "Invalid JSON: Expecting value: line 1 column 1 (char 0)"
]

Bad fake model score:
{
  "success": false,
  "expected_path": "plan.txt",
  "expected_content": "hello from the harness",
  "actual_content": null,
  "final_files": {}
}


In [5]:
# Empty scratch cell kept intentionally for your own quick experiments.


## Now Swap In BFCL

BFCL is the real benchmark we care about. The first file we will inspect is `BFCL_v3_multi_turn_base.json`.

Each line is one benchmark task. The shape is already close to our toy harness:

`question` is the user turns, `initial_config` is the simulator state, `path` is the expected tool path, and `involved_classes` tells us which API systems are active.

In [6]:
from urllib.request import urlretrieve

BFCL_REVISION = "70882e013073db2d6731d9f165a5eed992e28615"
BFCL_DATA_URL = (
    "https://huggingface.co/datasets/"
    "gorilla-llm/Berkeley-Function-Calling-Leaderboard/"
    f"raw/{BFCL_REVISION}/{BFCL_FILE}"
)

RAW_DATA_DIR = DATA_DIR / "raw" / "bfcl"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
bfcl_path = RAW_DATA_DIR / BFCL_FILE

if not bfcl_path.exists():
    print(f"Downloading {BFCL_FILE}...")
    urlretrieve(BFCL_DATA_URL, bfcl_path)
else:
    print(f"Using cached file: {bfcl_path}")

print(f"BFCL file: {bfcl_path}")
print(f"Size: {bfcl_path.stat().st_size:,} bytes")

Using cached file: /Users/kargarisaac/codes/personal/distillation-blogs/data/raw/bfcl/BFCL_v3_multi_turn_base.json
BFCL file: /Users/kargarisaac/codes/personal/distillation-blogs/data/raw/bfcl/BFCL_v3_multi_turn_base.json
Size: 391,058 bytes


The file is JSONL, not one giant JSON object. That means we read it line by line. This is common for benchmark files because it lets us stream examples without loading everything at once.

In [7]:
def load_jsonl(path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


bfcl_examples = load_jsonl(bfcl_path)
first_example = bfcl_examples[0]

print(f"Loaded {len(bfcl_examples)} BFCL examples.")
print("First example keys:", list(first_example.keys()))
print("First example id:", first_example["id"])
print("Involved API systems:", first_example["involved_classes"])
print("Expected tool path:")
print(json.dumps(first_example["path"], indent=2))

Loaded 200 BFCL examples.
First example keys: ['id', 'question', 'initial_config', 'path', 'involved_classes']
First example id: multi_turn_base_0
Involved API systems: ['TwitterAPI', 'GorillaFileSystem']
Expected tool path:
[
  "GorillaFileSystem.find",
  "GorillaFileSystem.mv",
  "GorillaFileSystem.grep",
  "GorillaFileSystem.sort",
  "GorillaFileSystem.diff",
  "TwitterAPI.post_tweet"
]


Now we inspect the row like a human. The goal is not to train yet. The goal is to see the benchmark contract: user turns, starting world state, and expected tools.

In [8]:
def show_bfcl_user_turns(example: dict[str, Any]) -> None:
    for turn_index, turn_messages in enumerate(example["question"], start=1):
        print(f"Turn {turn_index}")
        for message in turn_messages:
            print(f"  {message['role']}: {message['content']}")


def summarize_initial_config(example: dict[str, Any]) -> dict[str, Any]:
    summary: dict[str, Any] = {}
    for system_name, system_state in example["initial_config"].items():
        if isinstance(system_state, dict):
            summary[system_name] = list(system_state.keys())
        else:
            summary[system_name] = type(system_state).__name__
    return summary


show_bfcl_user_turns(first_example)
print("\nInitial config summary:")
print(json.dumps(summarize_initial_config(first_example), indent=2))

Turn 1
  user: Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory
Turn 2
  user: Perform a detailed search using grep to identify sections in the file pertaining to 'budget analysis'.
Turn 3
  user: Upon identifying the requisite 'budget analysis' content, sort the 'final_report.pdf' by line for improved clarity and comprehension.
Turn 4
  user: Move 'previous_report.pdf' in document directory to temp as well and having final report also there, proceed to juxtapose it with 'previous_report.pdf' to detect any critical alterations.

Initial config summary:
{
  "GorillaFileSystem": [
    "root"
  ],
  "TwitterAPI": [
    "tweet_counter",
    "tweets",
    "username",
    "password"
  ]
}


For this first example, the simulator state contains a fake file system and a fake Twitter API. The model has to operate on that state through tools. If it says "done" without calling tools, nothing in the environment changes, so the benchmark should fail it.

In [9]:
print(json.dumps(first_example["initial_config"], indent=2)[:3000])

{
  "GorillaFileSystem": {
    "root": {
      "workspace": {
        "type": "directory",
        "contents": {
          "document": {
            "type": "directory",
            "contents": {
              "final_report.pdf": {
                "type": "file",
                "content": "Year2024 This is the final report content including budget analysis and other sections."
              },
              "previous_report.pdf": {
                "type": "file",
                "content": "Year203 This is the previous report content with different budget analysis."
              }
            }
          },
          "archive": {
            "type": "directory",
            "contents": {}
          }
        }
      }
    }
  },
  "TwitterAPI": {
    "tweet_counter": 3,
    "tweets": {
      "0": {
        "id": 0,
        "username": "analyst_pro",
        "content": "Just finished analyzing the reports!",
        "tags": [
          "#analysis",
          "#reports"
        ],
    

In [10]:
# Empty scratch cell kept intentionally for your own quick experiments.


## Load The Tool Definitions

The benchmark row tells us which API systems are involved, but the model also needs tool definitions. BFCL keeps those in separate JSONL files. Each line is one callable tool with a name, description, parameter schema, and response schema.

This is the first place where the toy harness becomes real: the model will see tool docs, but it will not see the hidden simulator state. The hidden state lives inside the harness and is revealed only through tool responses.

In [11]:
GORILLA_REPO_COMMIT = "6ea57973c7a6097fd7c5915698c54c17c5b1b6c8"
FUNC_DOC_BASE_URL = (
    "https://raw.githubusercontent.com/ShishirPatil/gorilla/"
    f"{GORILLA_REPO_COMMIT}/"
    "berkeley-function-call-leaderboard/bfcl_eval/data/multi_turn_func_doc"
)

CLASS_TO_FUNC_DOC = {
    "GorillaFileSystem": "gorilla_file_system.json",
    "MathAPI": "math_api.json",
    "MessageAPI": "message_api.json",
    "TicketAPI": "ticket_api.json",
    "TradingBot": "trading_bot.json",
    "TravelAPI": "travel_booking.json",
    "TwitterAPI": "posting_api.json",
    "VehicleControlAPI": "vehicle_control.json",
}


def cache_function_doc_file(class_name: str) -> Path:
    file_name = CLASS_TO_FUNC_DOC[class_name]
    local_path = RAW_DATA_DIR / "multi_turn_func_doc" / file_name
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        urlretrieve(f"{FUNC_DOC_BASE_URL}/{file_name}", local_path)

    return local_path


def load_function_docs(class_name: str) -> list[dict[str, Any]]:
    path = cache_function_doc_file(class_name)
    tools = load_jsonl(path)
    for tool in tools:
        tool["bfcl_class"] = class_name
    return tools


def load_tools_for_example(example: dict[str, Any]) -> list[dict[str, Any]]:
    tools: list[dict[str, Any]] = []
    for class_name in example["involved_classes"]:
        tools.extend(load_function_docs(class_name))
    return tools


active_tools = load_tools_for_example(first_example)
expected_tool_names = [tool_path.split(".")[-1] for tool_path in first_example["path"]]

print(f"Loaded {len(active_tools)} tool definitions for this example.")
print("Expected tool names from the benchmark path:")
print(expected_tool_names)

Loaded 32 tool definitions for this example.
Expected tool names from the benchmark path:
['find', 'mv', 'grep', 'sort', 'diff', 'post_tweet']


The benchmark gives us the expected path with class names, like `GorillaFileSystem.mv`. The model usually emits the method name, like `mv`, and the harness maps that method back to the active API instance.

In [12]:
def summarize_tool(tool: dict[str, Any]) -> str:
    parameters = tool.get("parameters", {})
    properties = parameters.get("properties", {})
    required = parameters.get("required", [])
    params = ", ".join(properties.keys())
    return f"{tool['bfcl_class']}.{tool['name']}({params}) required={required}"


for tool in active_tools:
    marker = "*" if tool["name"] in expected_tool_names else " "
    print(f"{marker} {summarize_tool(tool)}")

  TwitterAPI.authenticate_twitter(username, password) required=['username', 'password']
  TwitterAPI.comment(tweet_id, comment_content) required=['tweet_id', 'comment_content']
  TwitterAPI.follow_user(username_to_follow) required=['username_to_follow']
  TwitterAPI.get_tweet(tweet_id) required=['tweet_id']
  TwitterAPI.get_tweet_comments(tweet_id) required=['tweet_id']
  TwitterAPI.get_user_stats(username) required=['username']
  TwitterAPI.get_user_tweets(username) required=['username']
  TwitterAPI.list_all_following() required=[]
  TwitterAPI.mention(tweet_id, mentioned_usernames) required=['tweet_id', 'mentioned_usernames']
* TwitterAPI.post_tweet(content, tags, mentions) required=['content']
  TwitterAPI.posting_get_login_status() required=[]
  TwitterAPI.retweet(tweet_id) required=['tweet_id']
  TwitterAPI.search_tweets(keyword) required=['keyword']
  TwitterAPI.unfollow_user(username_to_unfollow) required=['username_to_unfollow']
  GorillaFileSystem.cat(file_name) required=['fi

## Render A Simplified First Prompt

Before using the real tokenizer, we render a simplified prompt by hand. This is only for learning the pieces: tool schemas, user message, and assistant generation slot.

Do not treat this manual renderer as the final Qwen format. The real format comes from `tokenizer.apply_chat_template(...)` in the next section.


In [13]:
def tool_for_qwen_prompt(tool: dict[str, Any]) -> dict[str, Any]:
    clean_tool = {
        "name": tool["name"],
        "description": tool["description"],
        "parameters": tool["parameters"],
    }
    if "response" in tool:
        clean_tool["response"] = tool["response"]
    return clean_tool


def render_simplified_tool_prompt(messages: list[dict[str, str]], tools: list[dict[str, Any]]) -> str:
    tool_lines = "\n".join(
        json.dumps(tool_for_qwen_prompt(tool), ensure_ascii=False)
        for tool in tools
    )

    system = (
        "<|im_start|>system\n"
        "# Tools\n\n"
        "You may call one or more functions to assist with the user query.\n\n"
        "You are provided with function signatures within <tools></tools> XML tags:\n"
        "<tools>\n"
        f"{tool_lines}\n"
        "</tools>\n\n"
        "For each function call, return a JSON object with function name and arguments "
        "inside <tool_call></tool_call> XML tags:\n"
        "<tool_call>\n"
        "{\"name\": <function-name>, \"arguments\": <args-json-object>}\n"
        "</tool_call><|im_end|>\n"
    )

    rendered_messages = []
    for message in messages:
        rendered_messages.append(
            f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"
        )

    return system + "".join(rendered_messages) + "<|im_start|>assistant\n"


first_turn_messages = first_example["question"][0]
first_turn_prompt = render_simplified_tool_prompt(first_turn_messages, active_tools)

print(first_turn_prompt[:4000])
print("\n--- prompt continues ---")
print(f"Prompt length: {len(first_turn_prompt):,} characters")

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "authenticate_twitter", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twitter. Tool description: Authenticate a user with username and password.", "parameters": {"type": "dict", "properties": {"username": {"type": "string", "description": "Username of the user."}, "password": {"type": "string", "description": "Password of the user."}}, "required": ["username", "password"]}, "response": {"type": "dict", "properties": {"authentication_status": {"type": "boolean", "description": "True if authenticated, False otherwise."}}}}
{"name": "comment", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twit

Pause here and inspect the prompt. This manual renderer is useful for understanding the pieces, but it is not the source of truth. The source of truth is the tokenizer's native chat template.

## Use The Real Qwen Chat Template

Now we load the actual Qwen tokenizer and let it render the prompt.

This matters because tool-call formatting is part of what we are benchmarking and training. If we accidentally train on a made-up format, we are no longer testing Qwen's native tool-calling behavior.

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)

important_tokens = [
    "<tool_call>",
    "</tool_call>",
    "<tool_response>",
    "</tool_response>",
    "<think>",
    "</think>",
    "<|im_start|>",
    "<|im_end|>",
]

print(type(tokenizer).__name__)
print(f"Vocab size: {tokenizer.vocab_size:,}")
print("Important token ids:")
for token in important_tokens:
    print(f"  {token:16s} {tokenizer.convert_tokens_to_ids(token)}")

Qwen2Tokenizer
Vocab size: 248,044
Important token ids:
  <tool_call>      248058
  </tool_call>     248059
  <tool_response>  248066
  </tool_response> 248067
  <think>          248068
  </think>         248069
  <|im_start|>     248045
  <|im_end|>       248046


The token IDs are important later for logit distillation. If the teacher and student share the same tokenizer, we can compare token distributions directly. That is one of the reasons this project uses a same-family Qwen teacher and student.

In [15]:
def tool_for_chat_template(tool: dict[str, Any]) -> dict[str, Any]:
    clean_tool = tool_for_qwen_prompt(tool)

    # BFCL's docs use "dict". JSON-schema-style tool templates usually expect "object".
    # This is a schema normalization step, not a correctness shortcut.
    parameters = json.loads(json.dumps(clean_tool["parameters"]))
    if parameters.get("type") == "dict":
        parameters["type"] = "object"
    clean_tool["parameters"] = parameters

    return clean_tool


chat_template_tools = [tool_for_chat_template(tool) for tool in active_tools]

native_first_turn_prompt = tokenizer.apply_chat_template(
    first_turn_messages,
    tools=chat_template_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

tool_block_start = native_first_turn_prompt.index("<tools>")
format_instruction_start = native_first_turn_prompt.index("If you choose to call a function")
format_instruction_end = native_first_turn_prompt.index("<|im_end|>", format_instruction_start)

print("Tool definitions preview. These are JSON schemas inside <tools>:")
print(native_first_turn_prompt[tool_block_start:tool_block_start + 1500])
print("\n--- output-format instruction from the same prompt ---")
print(native_first_turn_prompt[format_instruction_start:format_instruction_end])
print("\n--- prompt length ---")
print(f"Native prompt length: {len(native_first_turn_prompt):,} characters")
print(f"Manual prompt length: {len(first_turn_prompt):,} characters")


Tool definitions preview. These are JSON schemas inside <tools>:
<tools>
{"name": "authenticate_twitter", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twitter. Tool description: Authenticate a user with username and password.", "parameters": {"type": "object", "properties": {"username": {"type": "string", "description": "Username of the user."}, "password": {"type": "string", "description": "Password of the user."}}, "required": ["username", "password"]}, "response": {"type": "dict", "properties": {"authentication_status": {"type": "boolean", "description": "True if authenticated, False otherwise."}}}}
{"name": "comment", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twitter. Tool description: Comment on a tweet for the authenticated user.", "parameters": {"type": "object"

Notice the distinction:

1. The tool definitions are JSON objects inside `<tools>...</tools>`. That is what you saw at the top of the rendered prompt.
2. The model is instructed to answer with XML-like `<tool_call>` blocks. That instruction appears after the tool list in the same system prompt.

So both are true. JSON describes the available tools. XML-like tags describe the model output we need to parse.

```text
<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
</function>
</tool_call>
```

That is the format we should parse and train on for Qwen3.5. The manual renderer was useful for learning the boundary. The tokenizer-rendered prompt is what we use for actual model calls.


In [16]:
native_prompt_token_ids = tokenizer(
    native_first_turn_prompt,
    add_special_tokens=False,
)["input_ids"]

print(f"Native first-turn prompt tokens: {len(native_prompt_token_ids):,}")
print("Last 20 token ids:")
print(native_prompt_token_ids[-20:])

Native first-turn prompt tokens: 5,984
Last 20 token ids:
[6, 6025, 303, 2128, 13, 7170, 2617, 310, 1791, 279, 6025, 248046, 198, 248045, 74455, 198, 248068, 271, 248069, 271]


## Run The Base Student Once

Before we write a parser, we should look at the raw model output. The tokenizer tells us the contract, but the first generation tells us what the base 0.8B model actually does.

This next cell loads `Qwen/Qwen3.5-0.8B`. The first run will download the model weights. On Apple Silicon, PyTorch should use the MPS backend if it is available.


In [17]:
import torch
from transformers import AutoModelForCausalLM

def pick_torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = pick_torch_device()
dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32

print(f"Device: {device}")
print(f"Dtype: {dtype}")


Device: mps
Dtype: torch.float16


In [18]:
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    dtype=dtype,
    trust_remote_code=True,
)
student_model.to(device)
student_model.eval()

print("Loaded student model.")


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded student model.


Now we generate a short continuation from the native first-turn prompt. We print the raw string exactly because this is the evidence we will use to understand the baseline failure mode.

At this stage, do not fix or repair the output. If it is malformed, that is a real baseline result.


In [19]:
print(native_first_turn_prompt)

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"name": "authenticate_twitter", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twitter. Tool description: Authenticate a user with username and password.", "parameters": {"type": "object", "properties": {"username": {"type": "string", "description": "Username of the user."}, "password": {"type": "string", "description": "Password of the user."}}, "required": ["username", "password"]}, "response": {"type": "dict", "properties": {"authentication_status": {"type": "boolean", "description": "True if authenticated, False otherwise."}}}}
{"name": "comment", "description": "This tool belongs to the TwitterAPI, which provides core functionality for posting tweets, retweeting, commenting, and following users on Twitter. Tool description: Comment on a tweet for the authenticated user.", "parameters": {"type": 

In [20]:
MAX_NEW_TOKENS = 2048
ASSISTANT_TURN_END_TOKEN = "<|im_end|>"
ASSISTANT_TURN_END_TOKEN_ID = tokenizer.convert_tokens_to_ids(ASSISTANT_TURN_END_TOKEN)
GENERATION_EOS_TOKEN_IDS = [
    token_id
    for token_id in [tokenizer.eos_token_id, ASSISTANT_TURN_END_TOKEN_ID]
    if isinstance(token_id, int) and token_id >= 0
]

inputs = tokenizer(
    native_first_turn_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.inference_mode():
    generated = student_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        eos_token_id=GENERATION_EOS_TOKEN_IDS,
        pad_token_id=tokenizer.eos_token_id,
    )

new_token_ids = generated[0, inputs["input_ids"].shape[-1]:]
raw_student_output = tokenizer.decode(new_token_ids, skip_special_tokens=False)

print("Raw student output:")
print(raw_student_output)


Raw student output:
<tool_call>
<function=cd>
<parameter=folder>
document
</parameter>
</function>
</tool_call><|im_end|>


After running this, inspect the output before moving on. We care about questions like: did it use `<tool_call>` tags, did it choose a plausible function, did it include required parameters, and did it stop cleanly?

Only after seeing the raw behavior will we write the parser and connect the output to the simulator. The parser will follow the Qwen template contract, not just this one generation.


## Parse Qwen Tool Calls

Now we turn raw model text into structured actions. This parser follows the Qwen chat-template contract, not one specific model output.

The model can include natural language before the first `<tool_call>`, but after a tool call it should stop. For now, we parse valid tool-call blocks and separately record parser errors.


In [21]:
@dataclass
class ParsedQwenToolCall:
    name: str
    arguments: dict[str, str]
    raw_block: str


@dataclass
class QwenParseResult:
    calls: list[ParsedQwenToolCall]
    errors: list[str]


def extract_tag_blocks(text: str, start_tag: str, end_tag: str) -> tuple[list[str], list[str]]:
    blocks: list[str] = []
    errors: list[str] = []
    cursor = 0

    while True:
        start = text.find(start_tag, cursor)
        if start == -1:
            break

        content_start = start + len(start_tag)
        end = text.find(end_tag, content_start)
        if end == -1:
            errors.append(f"Found {start_tag} without matching {end_tag}.")
            break

        blocks.append(text[content_start:end].strip())
        cursor = end + len(end_tag)

    return blocks, errors


def parse_qwen_tool_call_block(block: str) -> tuple[ParsedQwenToolCall | None, list[str]]:
    lines = [line.strip() for line in block.splitlines() if line.strip()]
    errors: list[str] = []

    if not lines:
        return None, ["Empty tool_call block."]

    function_line = lines[0]
    if not function_line.startswith("<function=") or not function_line.endswith(">"):
        return None, ["tool_call block must start with <function=name>."]

    function_name = function_line[len("<function="):-1].strip()
    if not function_name:
        errors.append("Function name is empty.")

    arguments: dict[str, str] = {}
    index = 1
    while index < len(lines):
        line = lines[index]

        if line == "</function>":
            trailing = lines[index + 1:]
            if trailing:
                errors.append(f"Unexpected content after </function>: {trailing}")
            break

        if not line.startswith("<parameter=") or not line.endswith(">"):
            errors.append(f"Expected <parameter=name>, got: {line}")
            index += 1
            continue

        parameter_name = line[len("<parameter="):-1].strip()
        if not parameter_name:
            errors.append("Parameter name is empty.")

        value_lines: list[str] = []
        index += 1
        while index < len(lines) and lines[index] != "</parameter>":
            value_lines.append(lines[index])
            index += 1

        if index >= len(lines):
            errors.append(f"Parameter {parameter_name} is missing </parameter>.")
            break

        if parameter_name:
            arguments[parameter_name] = "\n".join(value_lines).strip()

        index += 1
    else:
        errors.append("tool_call block is missing </function>.")

    if errors:
        return None, errors

    return ParsedQwenToolCall(name=function_name, arguments=arguments, raw_block=block), []


def parse_qwen_tool_calls(text: str) -> QwenParseResult:
    text = text.split("<|im_end|>", 1)[0]
    blocks, errors = extract_tag_blocks(text, "<tool_call>", "</tool_call>")
    calls: list[ParsedQwenToolCall] = []

    for block in blocks:
        call, block_errors = parse_qwen_tool_call_block(block)
        errors.extend(block_errors)
        if call is not None:
            calls.append(call)

    if not blocks:
        errors.append("No <tool_call> block found.")

    return QwenParseResult(calls=calls, errors=errors)


Test the parser on the raw output from the previous generation cell. This notebook is meant to be run sequentially, so each parse step consumes the model output from the step before it.


In [22]:
if "raw_student_output" not in globals():
    raise RuntimeError("Run the first generation cell before parsing the first action.")

parse_result = parse_qwen_tool_calls(raw_student_output)

print("Parsed calls:")
for call in parse_result.calls:
    print({"name": call.name, "arguments": call.arguments})

print("\nParser errors:")
print(parse_result.errors)


Parsed calls:
{'name': 'cd', 'arguments': {'folder': 'document'}}

Parser errors:
[]


For your observed output, the parser should produce one structured action:

```python
{"name": "cd", "arguments": {"folder": "document"}}
```

That is exactly the object the harness needs before it can execute the tool in the simulator.


## Execute The Parsed Tool Call

Now the parsed action becomes real. We send it to BFCL's official multi-turn simulator. The simulator owns the hidden state, executes the tool method, mutates state if needed, and returns an observation string.

To keep this notebook lightweight, we use the official `bfcl_eval` package for simulator code but install it without its full leaderboard dependency tree:

```bash
uv pip install --no-deps bfcl-eval
```


In [23]:
try:
    from bfcl_eval.eval_checker.multi_turn_eval.multi_turn_utils import execute_multi_turn_func_call
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install the lightweight official BFCL simulator package with: "
        "uv pip install --no-deps bfcl-eval"
    ) from exc


tool_schemas_by_name = {tool["name"]: tool for tool in active_tools}
if len(tool_schemas_by_name) != len(active_tools):
    raise ValueError("Duplicate tool names in active tool set; executor mapping would be ambiguous.")


def coerce_argument_value(value: str, schema: dict[str, Any]) -> Any:
    target_type = schema.get("type", "string")

    if target_type == "integer":
        return int(value)
    if target_type == "float":
        return float(value)
    if target_type == "boolean":
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes"}:
            return True
        if normalized in {"false", "0", "no"}:
            return False
        raise ValueError(f"Cannot coerce {value!r} to boolean.")
    if target_type in {"array", "object", "dict"}:
        return json.loads(value)
    return value


def qwen_call_to_bfcl_execution_string(call: ParsedQwenToolCall) -> str:
    if call.name not in tool_schemas_by_name:
        raise ValueError(f"Unknown tool name: {call.name}")

    tool_schema = tool_schemas_by_name[call.name]
    parameter_schemas = tool_schema.get("parameters", {}).get("properties", {})
    required_parameters = set(tool_schema.get("parameters", {}).get("required", []))
    provided_parameters = set(call.arguments.keys())

    missing = sorted(required_parameters - provided_parameters)
    if missing:
        raise ValueError(f"Missing required parameters for {call.name}: {missing}")

    coerced_arguments: dict[str, Any] = {}
    for name, value in call.arguments.items():
        if name not in parameter_schemas:
            raise ValueError(f"Unknown parameter for {call.name}: {name}")
        coerced_arguments[name] = coerce_argument_value(value, parameter_schemas[name])

    argument_text = ",".join(
        f"{name}={repr(value)}" for name, value in coerced_arguments.items()
    )
    return f"{call.name}({argument_text})"


In [24]:
parse_result

QwenParseResult(calls=[ParsedQwenToolCall(name='cd', arguments={'folder': 'document'}, raw_block='<function=cd>\n<parameter=folder>\ndocument\n</parameter>\n</function>')], errors=[])

In [25]:
if not parse_result.calls:
    raise ValueError(f"Cannot execute because parsing failed: {parse_result.errors}")

execution_calls = [
    qwen_call_to_bfcl_execution_string(call)
    for call in parse_result.calls
]

print("BFCL execution calls:")
print(execution_calls)


BFCL execution calls:
["cd(folder='document')"]


In [26]:
from uuid import uuid4

BFCL_RUN_ID = f"notebook_student_{uuid4().hex[:8]}"

execution_results, involved_instances = execute_multi_turn_func_call(
    func_call_list=execution_calls,
    initial_config=first_example["initial_config"],
    involved_classes=first_example["involved_classes"],
    model_name=BFCL_RUN_ID,
    test_entry_id=first_example["id"],
    long_context=False,
    is_evaL_run=False,
)

print("Execution results returned to the model:")
print(json.dumps(execution_results, indent=2))


Execution results returned to the model:
[
  "{\"current_working_directory\": \"document\"}"
]


For the observed first action, the simulator should return something like:

```json
["{\"current_working_directory\": \"document\"}"]
```

That observation is what the harness appends to the conversation before asking the model for the next action.


In [27]:
def strip_generated_special_tokens(text: str) -> str:
    cleaned = text.split("<|im_end|>", 1)[0].strip()
    for special_token in ["<|endoftext|>"]:
        cleaned = cleaned.replace(special_token, "")
    return cleaned.strip()


def render_tool_response_messages(execution_results: list[str]) -> list[dict[str, str]]:
    return [
        {"role": "tool", "content": result}
        for result in execution_results
    ]


assistant_action_message = {
    "role": "assistant",
    "content": strip_generated_special_tokens(raw_student_output),
}
tool_response_messages = render_tool_response_messages(execution_results)

messages_after_first_step = first_turn_messages + [assistant_action_message] + tool_response_messages

print("Conversation after first tool step:")
for message in messages_after_first_step:
    print("-", message["role"])
    print(message["content"])


Conversation after first tool step:
- user
Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory
- assistant
<tool_call>
<function=cd>
<parameter=folder>
document
</parameter>
</function>
</tool_call>
- tool
{"current_working_directory": "document"}


In [28]:
messages_after_first_step

[{'role': 'user',
  'content': "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"},
 {'role': 'assistant',
  'content': '<tool_call>\n<function=cd>\n<parameter=folder>\ndocument\n</parameter>\n</function>\n</tool_call>'},
 {'role': 'tool', 'content': '{"current_working_directory": "document"}'}]

Next, we will feed `messages_after_first_step` back through the tokenizer template and ask the model for step two. If the model continues with `mkdir(temp)` and `mv(final_report.pdf, temp)`, then the first output was a reasonable first step rather than a failure.


## Render Step Two Prompt

The harness loop is now visible: model action, simulator observation, then another model prompt. We render the next prompt before generating so you can inspect what the model will see at step two.


In [29]:
second_step_prompt = tokenizer.apply_chat_template(
    messages_after_first_step,
    tools=chat_template_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(second_step_prompt[-2500:])
print("\nSecond-step prompt tokens:", len(tokenizer(second_step_prompt, add_special_tokens=False)["input_ids"]))


e_name": {"type": "string", "description": "The name of the new file in the current directory. file_name is local to the current directory and does not allow path."}}, "required": ["file_name"]}, "response": {"type": "dict", "properties": {}}}
{"name": "wc", "description": "This tool belongs to the Gorilla file system. It is a simple file system that allows users to perform basic file operations such as navigating directories, creating files and directories, reading and writing to files, etc. Tool description: Count the number of lines, words, and characters in a file of any extension from current directory.", "parameters": {"type": "object", "properties": {"file_name": {"type": "string", "description": "Name of the file of current directory to perform wc operation on."}, "mode": {"type": "string", "description": "Mode of operation ('l' for lines, 'w' for words, 'c' for characters). ", "default": "l"}}, "required": ["file_name"]}, "response": {"type": "dict", "properties": {"count": {"

If `student_model` is still loaded from the earlier cell, you can generate the next action immediately. Otherwise, run the model-loading cell first.


In [30]:
if "student_model" not in globals():
    print("student_model is not loaded yet. Run the model-loading cell above first.")
else:
    second_inputs = tokenizer(
        second_step_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    second_inputs = {key: value.to(device) for key, value in second_inputs.items()}

    with torch.inference_mode():
        second_generated = student_model.generate(
            **second_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=GENERATION_EOS_TOKEN_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    second_new_token_ids = second_generated[0, second_inputs["input_ids"].shape[-1]:]
    raw_second_student_output = tokenizer.decode(second_new_token_ids, skip_special_tokens=False)

    print("Raw second student output:")
    print(raw_second_student_output)


Raw second student output:
<tool_call>
<function=mkdir>
<parameter=dir_name>
temp
</parameter>
</function>
</tool_call><|im_end|>


## Execute The Second Tool Step

The second action is exactly what we hoped for. After entering `document`, the model asks the environment to create `temp`. Now we parse and execute that action with the same BFCL run id, so the simulator remembers that the current directory is already `document`.


In [31]:
if "raw_second_student_output" not in globals():
    raise RuntimeError("Run the second generation cell before parsing the second action.")

second_parse_result = parse_qwen_tool_calls(raw_second_student_output)

print("Second-step parsed calls:")
for call in second_parse_result.calls:
    print(call)

print("\nSecond-step parser errors:")
print(second_parse_result.errors)


Second-step parsed calls:
ParsedQwenToolCall(name='mkdir', arguments={'dir_name': 'temp'}, raw_block='<function=mkdir>\n<parameter=dir_name>\ntemp\n</parameter>\n</function>')

Second-step parser errors:
[]


In [32]:
if "BFCL_RUN_ID" not in globals():
    raise RuntimeError("Run the first execution cell first, so BFCL has the state after cd(document).")
if not second_parse_result.calls:
    raise ValueError(f"Cannot execute because parsing failed: {second_parse_result.errors}")

second_execution_calls = [
    qwen_call_to_bfcl_execution_string(call)
    for call in second_parse_result.calls
]

second_execution_results, involved_instances = execute_multi_turn_func_call(
    func_call_list=second_execution_calls,
    initial_config=first_example["initial_config"],
    involved_classes=first_example["involved_classes"],
    model_name=BFCL_RUN_ID,
    test_entry_id=first_example["id"],
    long_context=False,
    is_evaL_run=False,
)

print("Second BFCL execution calls:")
print(second_execution_calls)
print("\nSecond execution results returned to the model:")
print(json.dumps(second_execution_results, indent=2))


Second BFCL execution calls:
["mkdir(dir_name='temp')"]

Second execution results returned to the model:
[
  "None"
]


If the simulator accepts `mkdir(temp)`, the next useful action should be to move `final_report.pdf` into that new directory. This is the little loop we care about: model chooses one action, environment changes state, model sees the observation, then chooses the next action.


In [33]:
second_assistant_action_message = {
    "role": "assistant",
    "content": strip_generated_special_tokens(raw_second_student_output),
}
second_tool_response_messages = render_tool_response_messages(second_execution_results)

messages_after_second_step = (
    messages_after_first_step
    + [second_assistant_action_message]
    + second_tool_response_messages
)

print("Conversation after second tool step:")
for message in messages_after_second_step:
    print("-", message["role"])
    print(message["content"][:500])


Conversation after second tool step:
- user
Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory
- assistant
<tool_call>
<function=cd>
<parameter=folder>
document
</parameter>
</function>
</tool_call>
- tool
{"current_working_directory": "document"}
- assistant
<tool_call>
<function=mkdir>
<parameter=dir_name>
temp
</parameter>
</function>
</tool_call>
- tool
None


## Render Step Three Prompt

Now we render the third prompt. At this point the model has seen two environment observations: it is inside `document`, and `temp` has been created there.


In [34]:
third_step_prompt = tokenizer.apply_chat_template(
    messages_after_second_step,
    tools=chat_template_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(third_step_prompt[-2500:])
print("\nThird-step prompt tokens:", len(tokenizer(third_step_prompt, add_special_tokens=False)["input_ids"]))


"type": "dict", "properties": {}}}
{"name": "wc", "description": "This tool belongs to the Gorilla file system. It is a simple file system that allows users to perform basic file operations such as navigating directories, creating files and directories, reading and writing to files, etc. Tool description: Count the number of lines, words, and characters in a file of any extension from current directory.", "parameters": {"type": "object", "properties": {"file_name": {"type": "string", "description": "Name of the file of current directory to perform wc operation on."}, "mode": {"type": "string", "description": "Mode of operation ('l' for lines, 'w' for words, 'c' for characters). ", "default": "l"}}, "required": ["file_name"]}, "response": {"type": "dict", "properties": {"count": {"type": "integer", "description": "The count of the number of lines, words, or characters in the file."}, "type": {"type": "string", "description": "The type of unit we are counting. [Enum]: [\"lines\", \"words

If the model is still loaded, generate the third action. This is the one that tells us whether the model can continue the plan beyond just navigation and directory creation.


In [35]:
if "student_model" not in globals():
    print("student_model is not loaded yet. Run the model-loading cell above first.")
else:
    third_inputs = tokenizer(
        third_step_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    third_inputs = {key: value.to(device) for key, value in third_inputs.items()}

    with torch.inference_mode():
        third_generated = student_model.generate(
            **third_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=GENERATION_EOS_TOKEN_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    third_new_token_ids = third_generated[0, third_inputs["input_ids"].shape[-1]:]
    raw_third_student_output = tokenizer.decode(third_new_token_ids, skip_special_tokens=False)

    print("Raw third student output:")
    print(raw_third_student_output)


KeyboardInterrupt: 

## Diagnose The Third Tool Step

This third action is the first interesting mistake. The model continues the task, but it does not follow the filesystem tool contract:

- It uses `cp`, but the user asked to move the file.
- It passes `document/final_report.pdf`, but the simulator is already inside `document`, and the tool description says the source must be local to the current directory.
- It passes `temp/final_report.pdf`, but the `destination` should also be local. Since `temp` is an existing directory, the clean move would be `mv(source='final_report.pdf', destination='temp')`.

Let's execute the model's real action rather than correcting it by hand. The simulator will tell us what actually happens.


In [ ]:
if "raw_third_student_output" not in globals():
    raise RuntimeError("Run the third generation cell before parsing the third action.")

third_parse_result = parse_qwen_tool_calls(raw_third_student_output)

print("Third-step parsed calls:")
for call in third_parse_result.calls:
    print(call)

print("\nThird-step parser errors:")
print(third_parse_result.errors)


Third-step parsed calls:
ParsedQwenToolCall(name='cp', arguments={'source': 'document/final_report.pdf', 'destination': 'temp/final_report.pdf'}, raw_block='<function=cp>\n<parameter=source>\ndocument/final_report.pdf\n</parameter>\n<parameter=destination>\ntemp/final_report.pdf\n</parameter>\n</function>')

Third-step parser errors:
[]


In [ ]:
if "BFCL_RUN_ID" not in globals():
    raise RuntimeError("Run the earlier execution cells first, so BFCL has the state after cd(document) and mkdir(temp).")
if not third_parse_result.calls:
    raise ValueError(f"Cannot execute because parsing failed: {third_parse_result.errors}")

third_execution_calls = [
    qwen_call_to_bfcl_execution_string(call)
    for call in third_parse_result.calls
]

third_execution_results, involved_instances = execute_multi_turn_func_call(
    func_call_list=third_execution_calls,
    initial_config=first_example["initial_config"],
    involved_classes=first_example["involved_classes"],
    model_name=BFCL_RUN_ID,
    test_entry_id=first_example["id"],
    long_context=False,
    is_evaL_run=False,
)

print("Third BFCL execution calls:")
print(third_execution_calls)
print("\nThird execution results returned to the model:")
print(json.dumps(third_execution_results, indent=2))


Third BFCL execution calls:
["cp(source='document/final_report.pdf',destination='temp/final_report.pdf')"]

Third execution results returned to the model:
[
  "{\"error\": \"cp: cannot copy 'document/final_report.pdf': No such file or directory\"}"
]


Notice what this tells us. The parser worked. The chat template worked. The simulator worked. The failure is the model's action choice: it chose the wrong tool and wrong argument shape for the current environment state.

For evaluation, this is exactly the kind of thing a benchmark can catch. For teaching, we can also feed the error back to the model and see whether it repairs itself.


In [ ]:
third_assistant_action_message = {
    "role": "assistant",
    "content": strip_generated_special_tokens(raw_third_student_output),
}
third_tool_response_messages = render_tool_response_messages(third_execution_results)

messages_after_third_step = (
    messages_after_second_step
    + [third_assistant_action_message]
    + third_tool_response_messages
)

print("Conversation after third tool step:")
for message in messages_after_third_step:
    print("-", message["role"])
    print(message["content"][:500])


Conversation after third tool step:
- user
Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory
- assistant
<tool_call>
<function=cd>
<parameter=folder>
document
</parameter>
</function>
</tool_call>
- tool
{"current_working_directory": "document"}
- assistant
<tool_call>
<function=mkdir>
<parameter=dir_name>
temp
</parameter>
</function>
</tool_call>
- tool
None
- assistant
The directory 'temp' has been created successfully. Now, let's move the 'final_report.pdf' file from the 'document' directory to the 'temp' directory.

<tool_call>
<function=cp>
<parameter=source>
document/final_report.pdf
</parameter>
<parameter=destination>
temp/final_report.pdf
</parameter>
</function>
</tool_call>
- tool
{"error": "cp: cannot copy 'document/final_report.pdf': No such file or directory"}


## Render Step Four Prompt

The fourth prompt includes the simulator error. If we generate from here, we are testing recovery: can the model notice the environment rejected its action and choose a better one?


In [ ]:
fourth_step_prompt = tokenizer.apply_chat_template(
    messages_after_third_step,
    tools=chat_template_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(fourth_step_prompt[-2500:])
print("\nFourth-step prompt tokens:", len(tokenizer(fourth_step_prompt, add_special_tokens=False)["input_ids"]))


e file of current directory to perform wc operation on."}, "mode": {"type": "string", "description": "Mode of operation ('l' for lines, 'w' for words, 'c' for characters). ", "default": "l"}}, "required": ["file_name"]}, "response": {"type": "dict", "properties": {"count": {"type": "integer", "description": "The count of the number of lines, words, or characters in the file."}, "type": {"type": "string", "description": "The type of unit we are counting. [Enum]: [\"lines\", \"words\", \"characters\"]"}}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_parameter_2>
This is the value for the second parameter
that can span
multiple lines
</parameter>
</function>
</tool_call>

<IMPORTANT>
Reminder:
- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_

Generate one more action only if you want to inspect recovery behavior. We are not pretending this repaired action is the original model trajectory; it is a separate question: can the model recover after an environment error?


In [ ]:
if "student_model" not in globals():
    print("student_model is not loaded yet. Run the model-loading cell above first.")
else:
    fourth_inputs = tokenizer(
        fourth_step_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    fourth_inputs = {key: value.to(device) for key, value in fourth_inputs.items()}

    with torch.inference_mode():
        fourth_generated = student_model.generate(
            **fourth_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=GENERATION_EOS_TOKEN_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    fourth_new_token_ids = fourth_generated[0, fourth_inputs["input_ids"].shape[-1]:]
    raw_fourth_student_output = tokenizer.decode(fourth_new_token_ids, skip_special_tokens=False)

    print("Raw fourth student output:")
    print(raw_fourth_student_output)


Raw fourth student output:
It seems the 'document' directory might not exist yet. Let me check the current directory contents to see if it exists.

<tool_call>
<function=ls>
<parameter=a>
True
</parameter>
</function>
</tool_call><|im_end|>


## Diagnose The Fourth Tool Step

The model is trying to recover, but notice the state confusion. It says the `document` directory might not exist, even though the simulator already reported that the current working directory is `document`. The `ls(a=True)` call is still useful because it asks the environment what is actually in the current directory.


In [ ]:
if "raw_fourth_student_output" not in globals():
    raise RuntimeError("Run the fourth generation cell before parsing the fourth action.")

fourth_parse_result = parse_qwen_tool_calls(raw_fourth_student_output)

print("Fourth-step parsed calls:")
for call in fourth_parse_result.calls:
    print(call)

print("\nFourth-step parser errors:")
print(fourth_parse_result.errors)


Fourth-step parsed calls:
ParsedQwenToolCall(name='ls', arguments={'a': 'True'}, raw_block='<function=ls>\n<parameter=a>\nTrue\n</parameter>\n</function>')

Fourth-step parser errors:
[]


In [ ]:
if "BFCL_RUN_ID" not in globals():
    raise RuntimeError("Run the earlier execution cells first, so BFCL has the current simulator state.")
if not fourth_parse_result.calls:
    raise ValueError(f"Cannot execute because parsing failed: {fourth_parse_result.errors}")

fourth_execution_calls = [
    qwen_call_to_bfcl_execution_string(call)
    for call in fourth_parse_result.calls
]

fourth_execution_results, involved_instances = execute_multi_turn_func_call(
    func_call_list=fourth_execution_calls,
    initial_config=first_example["initial_config"],
    involved_classes=first_example["involved_classes"],
    model_name=BFCL_RUN_ID,
    test_entry_id=first_example["id"],
    long_context=False,
    is_evaL_run=False,
)

print("Fourth BFCL execution calls:")
print(fourth_execution_calls)
print("\nFourth execution results returned to the model:")
print(json.dumps(fourth_execution_results, indent=2))


Fourth BFCL execution calls:
['ls(a=True)']

Fourth execution results returned to the model:
[
  "{\"current_directory_content\": [\"final_report.pdf\", \"previous_report.pdf\", \"temp\"]}"
]


This observation should help the model repair its next action. The environment says `final_report.pdf` and `temp` are both local names in the current directory, so the next clean action is still `mv(source='final_report.pdf', destination='temp')`.


In [ ]:
fourth_assistant_action_message = {
    "role": "assistant",
    "content": strip_generated_special_tokens(raw_fourth_student_output),
}
fourth_tool_response_messages = render_tool_response_messages(fourth_execution_results)

messages_after_fourth_step = (
    messages_after_third_step
    + [fourth_assistant_action_message]
    + fourth_tool_response_messages
)

print("Conversation after fourth tool step:")
for message in messages_after_fourth_step:
    print("-", message["role"])
    print(message["content"][:500])


Conversation after fourth tool step:
- user
Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory
- assistant
<tool_call>
<function=cd>
<parameter=folder>
document
</parameter>
</function>
</tool_call>
- tool
{"current_working_directory": "document"}
- assistant
<tool_call>
<function=mkdir>
<parameter=dir_name>
temp
</parameter>
</function>
</tool_call>
- tool
None
- assistant
The directory 'temp' has been created successfully. Now, let's move the 'final_report.pdf' file from the 'document' directory to the 'temp' directory.

<tool_call>
<function=cp>
<parameter=source>
document/final_report.pdf
</parameter>
<parameter=destination>
temp/final_report.pdf
</parameter>
</function>
</tool_call>
- tool
{"error": "cp: cannot copy 'document/final_report.pdf': No such file or directory"}
- assistant
It seems the 'document' directory might not exist yet. Let me check the current directory contents to see if it exists.

<tool_call>
<

## Render Step Five Prompt

The fifth prompt now includes a successful directory listing. This is a good place to see whether the model can recover from its earlier wrong path assumptions.


In [ ]:
fifth_step_prompt = tokenizer.apply_chat_template(
    messages_after_fourth_step,
    tools=chat_template_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(fifth_step_prompt[-2500:])
print("\nFifth-step prompt tokens:", len(tokenizer(fifth_step_prompt, add_special_tokens=False)["input_ids"]))


"string", "description": "The type of unit we are counting. [Enum]: [\"lines\", \"words\", \"characters\"]"}}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_parameter_2>
This is the value for the second parameter
that can span
multiple lines
</parameter>
</function>
</tool_call>

<IMPORTANT>
Reminder:
- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags
- Required parameters MUST be specified
- You may provide optional reasoning for your function call in natural language BEFORE the function call, but NOT after
- If there is no function call available, answer the question like normal with your current knowledge and do not tell the user about function calls
</IMPORTANT><|im_end|>
<|im_start|>user
Move 'final_report.pdf' within docu

Generate one more action if you want to inspect whether the model recovers. If it still does not choose the local-name `mv` call, that is strong baseline evidence for why distillation is useful here.


In [ ]:
if "student_model" not in globals():
    print("student_model is not loaded yet. Run the model-loading cell above first.")
else:
    fifth_inputs = tokenizer(
        fifth_step_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    fifth_inputs = {key: value.to(device) for key, value in fifth_inputs.items()}

    with torch.inference_mode():
        fifth_generated = student_model.generate(
            **fifth_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=GENERATION_EOS_TOKEN_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    fifth_new_token_ids = fifth_generated[0, fifth_inputs["input_ids"].shape[-1]:]
    raw_fifth_student_output = tokenizer.decode(fifth_new_token_ids, skip_special_tokens=False)

    print("Raw fifth student output:")
    print(raw_fifth_student_output)


Raw fifth student output:
I can see that the 'document' directory does not exist yet. The 'final_report.pdf' file is currently in the 'document' directory, but the 'temp' directory has not been created.

Let me create the 'temp' directory first, and then we can move the file.

<tool_call>
<function=mkdir>
<parameter=dir_name>
temp
</parameter>
</function>
</tool_call><|im_end|>


## Where We Are Now

We have seen enough from the manual trajectory. The small model can emit valid Qwen-style tool calls, and the harness can parse, execute, and feed observations back. But the model loses track of environment state. It repeats `mkdir(temp)` after `temp` already exists, uses path-like arguments where BFCL expects local names, and chooses `cp` when the user asked to move the file.

That is the baseline story. We should not keep repairing this example by hand. The next question is broader and more useful: how often does this model complete BFCL tasks correctly across many examples?


## Baseline Benchmark Harness

Now we turn the manual path into a reusable benchmark loop. The loop uses only generic rules:

- stop generation at `<|im_end|>`;
- stop a user turn when the model emits no parseable tool call;
- stop an example if it hits `max_steps_per_turn`;
- stop a turn after repeated simulator errors;
- score with BFCL's official multi-turn checker.

For scoring consistency, this section uses the BFCL v4 multi-turn base files installed with `bfcl_eval`, including the matching `possible_answer` file. The full split has 200 examples. For this first blog, we will use 150 examples to create distillation traces and hold out 50 examples for evaluation.

One naming note: `MAX_STEPS_PER_TURN` is the maximum number of tool-action rounds inside one user turn. It is not the number of benchmark examples or user turns. We set it to 10 so longer but reasonable tool trajectories can finish before the harness gives up.


## Student Eval With Shared Utils

The cells above are the teaching path: toy harness, BFCL inspection, prompt rendering, raw student generation, parser, simulator, and manual multi-step debugging.

For the actual 50-example baseline score, we now use the shared harness in `utils.py`. This keeps the runnable benchmark code in one place while preserving the teaching cells above.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

cwd = Path.cwd()
if (cwd / "utils.py").exists():
    blog_dir = cwd
elif (cwd / "1-distilling-a-0-8b-tool-calling-agent" / "utils.py").exists():
    blog_dir = cwd / "1-distilling-a-0-8b-tool-calling-agent"
else:
    raise RuntimeError("Run this notebook from the repo root or the blog folder.")

if str(blog_dir) not in sys.path:
    sys.path.insert(0, str(blog_dir))

import utils as u

u = importlib.reload(u)
split = u.load_bfcl_split()

print("Full BFCL v4 multi-turn base size:", len(split.all_entries))
print("Train split:", len(split.train_entries))
print("Held-out eval split:", len(split.eval_entries))
print("Current eval run:", len(split.benchmark_entries))
print("First train id:", split.train_entries[0]["id"])
print("First eval id:", split.benchmark_entries[0]["id"])
print("MAX_STEPS_PER_TURN:", u.MAX_STEPS_PER_TURN)
print("MAX_CONSECUTIVE_EXECUTION_ERRORS:", u.MAX_CONSECUTIVE_EXECUTION_ERRORS)


Full BFCL v4 multi-turn base size: 200
Train split: 150
Held-out eval split: 50
Current eval run: 50
First train id: multi_turn_base_0
First eval id: multi_turn_base_150
MAX_STEPS_PER_TURN: 20
MAX_CONSECUTIVE_EXECUTION_ERRORS: None


### Inspect One Held-Out Eval Example

This is the split we use for reporting before and after distillation. We do not train on these 50 examples.


In [ ]:
eval_example = split.benchmark_entries[0]
eval_answer = split.benchmark_answers[0]
eval_tools = u.load_package_tools_for_example(eval_example)

print("Example id:", eval_example["id"])
print("Involved systems:", eval_example["involved_classes"])
print("User turns:")
print(json.dumps(eval_example["question"], indent=2)[:3000])
print("\nGround truth tool path:")
print(json.dumps(eval_answer["ground_truth"], indent=2))
print("\nAvailable tool count:", len(eval_tools))
print("First 10 tools:", [tool["name"] for tool in eval_tools[:10]])


Example id: multi_turn_base_150
Involved systems: ['TravelAPI']
User turns:
[
  [
    {
      "role": "user",
      "content": "Wanderlust is calling, and I'm mapping out my travel from Rivermist to Stonebrook in the month of October\u2014the skies above must know my next destination! Share with me, if you will, what coin I must trade for a humble economy class seat on the 6th day of October in 2026."
    }
  ],
  [
    {
      "role": "user",
      "content": "Let's anchor my daily spend at 15400 GBP\u2014an expedient conversion\u2014per my esteemed authorization. I appreciate your assistance in tethering my budget to this new mooring. My access token is 'abc123token'."
    }
  ]
]

Ground truth tool path:
[
  [
    "get_flight_cost(travel_from='RMS', travel_to='SBK', travel_date='2026-10-06', travel_class='economy')"
  ],
  [
    "compute_exchange_rate(base_currency='GBP', target_currency='USD', value=15400.0)",
    "set_budget_limit(access_token='abc123token', budget_limit=22000.0)"

### Load Student Model For The Shared Harness

If you already loaded the model in the earlier teaching cells, this reloads it through the shared utility path so the eval cell is independent and reproducible.


In [ ]:
student_model, tokenizer, device = u.load_student_model_and_tokenizer()
generate_student_text = u.make_student_text_generator(student_model, tokenizer, device)
print("Loaded student on device:", device)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded student on device: mps


### Inspect One Full Student Trace Through The Shared Harness

The manual cells above walk through individual steps. This cell shows the same idea through the reusable benchmark harness before we run all 50 eval examples.


In [ ]:
teaching_entry = split.train_entries[0]
teaching_answer = split.train_answers[0]
teaching_trace = u.run_bfcl_entry_harness(
    teaching_entry,
    run_name="student_teaching_trace",
    tokenizer=tokenizer,
    generate_text=generate_student_text,
    capture_prompts=True,
)
teaching_score = u.score_bfcl_trace(
    teaching_entry,
    teaching_answer,
    teaching_trace["decoded_turns"],
    run_name="student_teaching_trace",
)

print("Teaching example:", teaching_entry["id"])
print("Score:", u.summarize_score(teaching_score))
print(json.dumps(u.make_json_safe(teaching_score), indent=2))
print()
print("Ground truth:")
print(json.dumps(teaching_answer["ground_truth"], indent=2))
print()
print("Student trace:")
for row in teaching_trace["trace"]:
    print(f"turn={row.get('turn')} step={row.get('step')} stop={row.get('stop_reason')}")
    if row.get("execution_calls"):
        print("  calls:", row["execution_calls"])
    if row.get("execution_results"):
        print("  observations:", row["execution_results"])
    if row.get("stop_reason") != "executed_tool_call":
        print("  raw output:", row.get("raw_output", "")[:500])


Teaching example: multi_turn_base_0
Score: multi_turn:instance_state_mismatch
{
  "valid": false,
  "error_message": "Model instance for GorillaFileSystem does not match the state with ground truth instance.",
  "error_type": "multi_turn:instance_state_mismatch",
  "details": {
    "differences": {
      "root": {
        "model": "<Directory: workspace, Parent: None, Contents: {'document': <Directory: document, Parent: workspace, Contents: {'final_report.pdf': <<File: final_report.pdf, Content: Year2024 This is the final report content including budget analysis and other sections.>>, 'previous_report.pdf': <<File: previous_report.pdf, Content: Year203 This is the previous report content with different budget analysis.>>, 'temp': <Directory: temp, Parent: document, Contents: {}>}>, 'archive': <Directory: archive, Parent: workspace, Contents: {}>}>",
        "ground_truth": "<Directory: workspace, Parent: None, Contents: {'document': <Directory: document, Parent: workspace, Contents: {'

### Run Student Eval On 50 Held-Out Examples: Pass@1

This is the main baseline metric for the blog. We give the student one chance per held-out BFCL task and score the full trajectory with the same parser, simulator, and BFCL scorer.

This is the number we should compare before and after distillation. If the student improves here, the model itself got better at choosing the next action.

The output file is also the resume cache. After each completed task, the result is written to disk. If the notebook stops halfway through, rerunning this cell skips cached task ids and continues from the missing ones.


In [ ]:
student_eval_path = u.OUTPUT_DIR / "qwen_0_8b_bfcl_v4_multi_turn_base_eval_qwen_parser_v2_full_trace_pass1.json"
LOG_TRACES_TO_MLFLOW = True
mlflow_config = u.MlflowConfig(enabled=LOG_TRACES_TO_MLFLOW)

print("MLflow logging enabled:", mlflow_config.enabled)
print("MLflow tracking URI:", mlflow_config.tracking_uri)
print("MLflow experiment:", mlflow_config.experiment_name)

student_eval = u.evaluate_entries(
    split.benchmark_entries,
    split.benchmark_answers,
    tokenizer=tokenizer,
    run_name_prefix="qwen_0_8b_student_eval_pass1",
    generate_text=generate_student_text,
    output_path=student_eval_path,
    capture_prompts=True,
    mlflow_config=mlflow_config,
    mlflow_tags={"model_role": "student", "model_name": STUDENT_MODEL},
)

print("\nStudent pass@1 accuracy:", student_eval["accuracy"])
print(f"Correct: {student_eval['correct_count']}/{student_eval['total_count']}")
print("Saved to:", student_eval_path)


### Optional Student Adaptive Pass@5 Diagnostic

This is **not** the main distillation metric. Adaptive pass@5 measures the student plus a retry/search policy, so it is a system-level number, not pure model ability.

We keep it as an optional diagnostic because it tells us whether sampling plus BFCL scoring can find correct trajectories. The retry strategy came from a small experiment: fixed retries replayed the same failures, seed-only at temperature `0.2` did not help, a temperature ramp `[0.2, 0.4, 0.6, 0.8, 1.0]` rescued one task in the first 10-train-task probe, and opening the sampler further made loops worse.


In [ ]:
RUN_STUDENT_ADAPTIVE_EVAL = False

if not RUN_STUDENT_ADAPTIVE_EVAL:
    print("Skipped optional student adaptive pass@5 eval. Set RUN_STUDENT_ADAPTIVE_EVAL = True to run it.")
else:
    student_adaptive_eval_path = u.OUTPUT_DIR / "qwen_0_8b_bfcl_v4_multi_turn_base_eval_adaptive_pass5.json"
    student_attempt_preview = u.adaptive_temperature_retry_attempts_for_task(0)
    print("Student adaptive eval retry attempts per task:", len(student_attempt_preview))
    print("Retry schedule:")
    print(json.dumps([u.attempt_config_to_dict(attempt) for attempt in student_attempt_preview], indent=2))

    generate_student_text_factory = u.make_student_text_generator_factory(student_model, tokenizer, device)
    student_adaptive_eval = u.evaluate_entries(
        split.benchmark_entries,
        split.benchmark_answers,
        tokenizer=tokenizer,
        run_name_prefix="qwen_0_8b_student_eval_adaptive_pass5",
        generate_text_factory=generate_student_text_factory,
        attempt_configs_factory=u.adaptive_temperature_retry_attempts_for_task,
        output_path=student_adaptive_eval_path,
        capture_prompts=True,
    )

    print("\nStudent adaptive pass@5 accuracy:", student_adaptive_eval["accuracy"])
    print(f"Correct: {student_adaptive_eval['correct_count']}/{student_adaptive_eval['total_count']}")
    print(f"First-attempt correct: {student_adaptive_eval['first_attempt_correct_count']}/{student_adaptive_eval['total_count']}")
    print("Saved to:", student_adaptive_eval_path)


### Recorded Student Baseline

The previous completed student pass@1 run on the 50 held-out BFCL eval tasks scored **4/50** correct, or **8% accuracy**. The JSON artifact was removed during output cleanup, so this note records the observed baseline result without rerunning the model.
